# **Access to Safe Drinking Water is Essential for Health**

#### Access to safe drinking water is a fundamental human right and a cornerstone of global public health. Ensuring water potability not only protects communities from waterborne diseases but also drives significant economic benefits. 
#### In this project, I will analyze a comprehensive dataset containing water quality metrics for 3,276 different water sources. By examining key chemical and physical parameters—such as pH, hardness, sulfate levels, and turbidity— I aim to perform a thorough exploratory data analysis, address missing values, and ultimately understand the defining characteristics that classify water as safe for human consumption.



In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


#### The file water_potability.csv contains water quality criteria for 3,276 different water sources. 

## 1. **Data Inspection:**

In [ ]:
data = pd.read_csv('water_potability.csv')
data

#### To see a concise summary of the dataframe's columns, data types, and non-null values:

In [ ]:
data.info()

In [ ]:
df = pd.DataFrame(data)

In [ ]:
df.describe(include = 'all')

In [ ]:
df.isna().sum()

#### For a better vision, we will use a custom function to clearly display the variables alongside their total counts, unique values, and the exact number of missing entries:

In [ ]:
def describe(df):

    variables = []
    count = []
    unique = []
    missing_values = []

    for col in df.columns:
        variables.append(col)
        count.append(df[col].count())
        unique.append(len(df[col].unique()))
        missing_values.append(df[col].isna().sum())

    result = pd.DataFrame({
        'variables' : variables,
        'count' : count,
        'unique' : unique,
        'missing_values' : missing_values
    })

    return result


describe(df)

#### Since there are more than 1000 missing values, two approaches are possible:

## 2. **Handling the missing values:**

### 2.1. **Deleting the missing values:**

In [ ]:
df_cleaned = df.dropna()
describe(df_cleaned)

#### The number of deleted missing values are not justifiable.

### **2.1 Imputing the missing values:** (Selected approach)

In [ ]:
def simple_knn_impute(df, target_col, k=10, debug=True ,debug_samples=2):
    # df = main data | target_col = with missing value | k = number of neighbors | debug = it's just a flag for me :)
    df = df.copy() 
    feature_cols = [
        col for col in df.columns # all of columns
        if col != target_col and df[col].isna().sum() == 0 # target and other missing values should be ingnore
    ]

    shown = 0 # For shown some rows 

    known = df[df[target_col].notna()] 
    missing = df[df[target_col].isna()] 

    for idx, row in missing.iterrows(): # idx = index | row = the row
        
        diffs = known[feature_cols].sub(row[feature_cols]).abs()

        distances = diffs.sum(axis=1) # Manhattan distance (L1)

        nearest_idx = distances.nsmallest(k).index # k nearest neighbors (KNN)

        imputed_value = known.loc[nearest_idx, target_col].mean() # Mean of knn

        df.at[idx, target_col] = imputed_value # The null value = now has a value

        if debug and shown < debug_samples:
            print(f"Missing sample index: {idx}")
            print("Missing row:")
            display(df.loc[[idx]])

            print("Nearest neighbors:")
            display(df.loc[nearest_idx])

            print(f"Imputed value for '{target_col}': {imputed_value:.4f}")
            print("Nearest neighbors indices:")
            print(list(nearest_idx))

            print("\nNeighbor values:")
            print(known.loc[nearest_idx, target_col].values)

            shown += 1

    return df

In [ ]:
df_imputed = simple_knn_impute(df, 'Sulfate', k=10, debug_samples=2)

In [ ]:
describe(df_imputed)

#### We repeated this process for the other features with missing values:

In [ ]:
df_imputed = simple_knn_impute(df_imputed, 'ph', k=10, debug_samples=2)
df_imputed = simple_knn_impute(df_imputed, 'Trihalomethanes', k=10, debug_samples=2)


#### Now we have a DataFrame without missing values:

In [ ]:
describe(df_imputed)

## 3. **Data Visualization**

### 3.1. **pH vs. Hardness**

#### **pH Value:**  
#### pH is an important parameter for evaluating the acid-base balance in water. It indicates whether water is acidic or alkaline. The World Health Organization (WHO) has declared the allowable range (marked with dashed line) for pH to be between 6.5 and 8.5. In the current investigation, the pH range was between 6.52 and 6.83, which falls within the standard WHO range.

#### **Hardness:** 
#### Water hardness is primarily caused by the presence of calcium and magnesium salts. These salts dissolve from earth layers along the path of water flow; the longer the contact time, the higher the hardness of the raw water. Historically, hardness was defined by the water's ability to precipitate soap due to these minerals.

In [ ]:
# Extract data columns for plotting
x = df_imputed['ph']
y = df_imputed['Hardness']
potability = df_imputed['Potability']

# Map potability values to colors (red for not potable, blue for potable)
colors = []

for p in potability:
    if p == 0:
        colors.append('red')
    else:
        colors.append('blue')

# Create figure and axis
fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

# Define pH boundaries for water potability zones
ph_max_potability = 8.5
ph_min_potability = 6.5  

# Shade regions outside the acceptable pH range (non-potable zones)
ax.axvspan(ph_max_potability, 15, alpha=0.15, color='gray', zorder=0)  # Right of line 2
ax.axvspan(-1, ph_min_potability, alpha=0.15, color='gray', zorder=0)  # Left of line 2

# Draw vertical boundary lines for pH thresholds
ax.axvline(x=ph_max_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_max_potability}')
ax.axvline(x=ph_min_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_min_potability}')

# Plot the data points
ax.scatter(x, y, marker='o', s=20, alpha=0.6, c=colors, zorder=2) 

# Configure axis limits and labels
ax.set_xlim(-1, 15)
ax.set_xlabel('pH', fontsize=12)
ax.set_ylabel('Hardness (ppm)', fontsize=12)
ax.set_title('Water Quality: pH vs Hardness', fontsize=14)
ax.grid(True, alpha=0.3)

# Add legend to distinguish potable vs non-potable water samples
legend_elements = [Patch(facecolor='red', label='Not Potable (0)'),
                   Patch(facecolor='blue', label='Potable (1)'),
                   Patch(facecolor='orange', label='Allowable pH Range')]
ax.legend(handles=legend_elements, loc='best')

plt.tight_layout()
plt.show()


### 3.2. **pH vs. TDS**

#### **Total Dissolved Solids (TDS):** 
#### Water can dissolve various organic and inorganic minerals such as potassium, calcium, sodium, bicarbonates, chlorides, magnesium, and sulfates. These substances can impart an undesirable taste or dilute color to the water. TDS is a key indicator of water quality; water with high TDS is considered highly mineralized. The desirable TDS limit is 500 mg/L, with a maximum allowable limit for drinking set at 1000 mg/L.

In [ ]:
# Visualize pH vs Total Dissolved Solids (TDS) with water potability indicators
# Two subplots: full range view and zoomed view of acceptable ranges

# Extract feature columns for visualization
x = df_imputed['ph']
y = df_imputed['Solids']
potability = df_imputed['Potability']

# Color mapping: red = non-potable water (0), blue = potable water (1)
colors = []

for p in potability:
    if p == 0:
        colors.append('red')
    else:
        colors.append('blue')

# Initialize figure with two vertically stacked subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12) , dpi=100)

# Define acceptable water quality thresholds
ph_max_potability = 8.5
ph_min_potability = 6.5  

TDS_max_potability = 1000
TDS_min_potability = 500 

# === SUBPLOT 1: Full range view ===

# Highlight non-potable pH zones (outside 6.5-8.5 range)
ax1.axvspan(ph_max_potability, 15, alpha=0.15, color='gray', zorder=0)  # High pH zone
ax1.axvspan(-1, ph_min_potability, alpha=0.15, color='gray', zorder=0)  # Low pH zone

# Highlight non-potable TDS zones (outside 500-1000 ppm range)
ax1.axhspan(0, TDS_min_potability, alpha=0.15, color='gray', zorder=0)  # Low TDS zone
ax1.axhspan(TDS_max_potability, 70000, alpha=0.15, color='gray', zorder=0)  # High TDS zone

# Draw pH threshold boundary lines
ax1.axvline(x=ph_max_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_max_potability}')
ax1.axvline(x=ph_min_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_min_potability}')

# Draw TDS threshold boundary lines
ax1.axhline(y=TDS_max_potability, color='green', linestyle='--', linewidth=2, label=f'TDS = {TDS_max_potability}')
ax1.axhline(y=TDS_min_potability, color='green', linestyle='--', linewidth=2, label=f'TDS = {TDS_min_potability}')

# Scatter plot of all water samples
ax1.scatter(x, y, marker='o', s=20, alpha=0.6, c=colors, zorder=2) 

# Configure plot appearance for full view
ax1.set_xlim(-1, 15)
ax1.set_ylim(0, 65000)
ax1.set_xlabel('pH', fontsize=12)
ax1.set_ylabel('Solids (TDS)', fontsize=12)
ax1.set_title('Water Quality: pH vs Solids', fontsize=14)
ax1.grid(True, alpha=0.3)

# Add color legend for potability classification
legend_elements = [Patch(facecolor='red', label='Not Potable (0)'),
                   Patch(facecolor='blue', label='Potable (1)')]
ax1.legend(handles=legend_elements, loc='best')

# === SUBPLOT 2: Zoomed view of acceptable ranges ===

# Highlight non-potable pH zones
ax2.axvspan(ph_max_potability, 15, alpha=0.15, color='gray', zorder=0)
ax2.axvspan(-1, ph_min_potability, alpha=0.15, color='gray', zorder=0)

# Highlight non-potable TDS zones
ax2.axhspan(0, TDS_min_potability, alpha=0.15, color='gray', zorder=0)
ax2.axhspan(TDS_max_potability, 70000, alpha=0.15, color='gray', zorder=0)

# Draw pH threshold boundary lines
ax2.axvline(x=ph_max_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_max_potability}')
ax2.axvline(x=ph_min_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_min_potability}')

# Draw TDS threshold boundary lines
ax2.axhline(y=TDS_max_potability, color='green', linestyle='--', linewidth=2, label=f'TDS = {TDS_max_potability}')
ax2.axhline(y=TDS_min_potability, color='green', linestyle='--', linewidth=2, label=f'TDS = {TDS_min_potability}')

# Scatter plot of all water samples
ax2.scatter(x, y, marker='o', s=20, alpha=0.6, c=colors, zorder=2) 

# Configure plot appearance for zoomed view
ax2.set_xlim(5, 9)
ax2.set_ylim(0, 1100)
ax2.set_xlabel('pH', fontsize=12)
ax2.set_ylabel('Solids (TDS)', fontsize=12)
ax2.set_title('Water Quality: pH vs Solids', fontsize=14)
ax2.grid(True, alpha=0.3)

# Add color legend for potability classification
legend_elements = [Patch(facecolor='red', label='Not Potable (0)'),
                   Patch(facecolor='blue', label='Potable (1)')]
ax2.legend(handles=legend_elements, loc='best')

# Adjust spacing between subplots and display
plt.tight_layout(h_pad=5.0)
plt.show()


### 3.3. **pH vs. Chloramines:**

#### **Chloramines:**
#### Chlorine and chloramines are the primary disinfectants used in municipal water systems. Chloramine typically forms when ammonia is added to chlorine to disinfect drinking water. Chlorine levels up to 4 mg/L (or 4 ppm) in drinking water are considered safe.

In [ ]:
# Visualize pH vs Chloramines with water potability indicators

# Extract feature columns for visualization
x = df_imputed['ph']
y = df_imputed['Chloramines']
potability = df_imputed['Potability']

# Color mapping: red = non-potable water (0), blue = potable water (1)
colors = []

for p in potability:
    if p == 0:
        colors.append('red')
    else:
        colors.append('blue')

# Initialize figure and axis
fig, (ax) = plt.subplots(figsize=(10, 6))

# Define acceptable water quality thresholds
ph_max_potability = 8.5
ph_min_potability = 6.5  

chloramines_max_potability = 4

# Highlight non-potable pH zones (outside 6.5-8.5 range)
ax.axvspan(ph_max_potability, 15, alpha=0.15, color='gray', zorder=0)  # High pH zone
ax.axvspan(-1, ph_min_potability, alpha=0.15, color='gray', zorder=0)  # Low pH zone

# Highlight non-potable chloramines zone (above 4 ppm)
ax.axhspan(chloramines_max_potability, 14, alpha=0.3, color='gray', zorder=0)  # High chloramines zone

# Draw pH threshold boundary lines
ax.axvline(x=ph_max_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_max_potability}')
ax.axvline(x=ph_min_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_min_potability}')

# Draw chloramines threshold boundary line
ax.axhline(y=chloramines_max_potability, color='green', linestyle='--', linewidth=2, label=f'Chloramines = {chloramines_max_potability}')

# Scatter plot of all water samples
ax.scatter(x, y, marker='o', s=20, alpha=0.6, c=colors, zorder=2) 

# Configure plot appearance
ax.set_xlim(-1, 15)
ax.set_ylim(0, 14)
ax.set_xlabel('pH', fontsize=12)
ax.set_ylabel('Chloramines', fontsize=12)
ax.set_title('Water Quality: pH vs Chloramines', fontsize=14)
ax.grid(True, alpha=0.3)

# Add color legend for potability classification
legend_elements = [Patch(facecolor='red', label='Not Potable (0)'),
                   Patch(facecolor='blue', label='Potable (1)')]
ax.legend(handles=legend_elements, loc='best')

# Display the plot
plt.tight_layout()
plt.show()


### 3.4. **pH vs. Sulfate:**

#### **Sulfate:**
#### Sulfates occur naturally in soil and rock minerals and are found in the air, groundwater, plants, and food. Their primary commercial use is in the chemical industry. Sulfate concentration in seawater is about 2700 mg/L, while most freshwater sources range between 3 to 30 mg/L, though some areas report up to 1000 mg/L.


In [ ]:
# Visualize pH vs Sulfate with water potability indicators

# Extract feature columns for visualization
x = df_imputed['ph']
y = df_imputed['Sulfate']
potability = df_imputed['Potability']

# Color mapping: red = non-potable water (0), blue = potable water (1)
colors = []

for p in potability:
    if p == 0:
        colors.append('red')
    else:
        colors.append('blue')

# Initialize figure and axis
fig, (ax) = plt.subplots(figsize=(10, 6))

# Define acceptable water quality thresholds
ph_max_potability = 8.5
ph_min_potability = 6.5  

# Highlight non-potable pH zones (outside 6.5-8.5 range)
ax.axvspan(ph_max_potability, 15, alpha=0.15, color='gray', zorder=0)  # High pH zone
ax.axvspan(-1, ph_min_potability, alpha=0.15, color='gray', zorder=0)  # Low pH zone

# Draw pH threshold boundary lines
ax.axvline(x=ph_max_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_max_potability}')
ax.axvline(x=ph_min_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_min_potability}')

# Scatter plot of all water samples
ax.scatter(x, y, marker='o', s=20, alpha=0.6, c=colors, zorder=2) 

# Configure plot appearance
ax.set_xlim(-1, 15)
ax.set_xlabel('pH', fontsize=12)
ax.set_ylabel('Sulfate', fontsize=12)
ax.set_title('Water Quality: pH vs Sulfate', fontsize=14)
ax.grid(True, alpha=0.3)

# Add color legend for potability classification
legend_elements = [Patch(facecolor='red', label='Not Potable (0)'),
                   Patch(facecolor='blue', label='Potable (1)')]
ax.legend(handles=legend_elements, loc='best')

# Display the plot
plt.tight_layout()
plt.show()


### 3.5. **pH vs. EC:**

#### **Electrical Conductivity (EC):** 
#### Pure water is a poor conductor of electricity and acts as an insulator. An increase in ion concentration increases the electrical conductivity. Generally, the amount of dissolved solids determines the conductivity level. EC measures the solution's ability to transmit an electric current. According to WHO standards, EC should not exceed 400 μS/cm.

In [ ]:
# Visualize pH vs Conductivity with water potability indicators

# Extract feature columns for visualization
x = df_imputed['ph']
y = df_imputed['Conductivity']
potability = df_imputed['Potability']

# Color mapping: red = non-potable water (0), blue = potable water (1)
colors = []

for p in potability:
    if p == 0:
        colors.append('red')
    else:
        colors.append('blue')

# Initialize figure and axis
fig, (ax) = plt.subplots(figsize=(10, 6))

# Define acceptable water quality thresholds
ph_max_potability = 8.5
ph_min_potability = 6.5  

conductivity_max_potability = 400

# Highlight non-potable pH zones (outside 6.5-8.5 range)
ax.axvspan(ph_max_potability, 15, alpha=0.15, color='gray', zorder=0)  # High pH zone
ax.axvspan(-1, ph_min_potability, alpha=0.15, color='gray', zorder=0)  # Low pH zone

# Highlight non-potable conductivity zone (above 400 µS/cm)
ax.axhspan(conductivity_max_potability, 800, alpha=0.3, color='gray', zorder=0)  # High conductivity zone

# Draw pH threshold boundary lines
ax.axvline(x=ph_max_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_max_potability}')
ax.axvline(x=ph_min_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_min_potability}')

# Draw conductivity threshold boundary line
ax.axhline(y=conductivity_max_potability, color='green', linestyle='--', linewidth=2, label=f'Conductivity = {conductivity_max_potability}')

# Scatter plot of all water samples
ax.scatter(x, y, marker='o', s=20, alpha=0.6, c=colors, zorder=2) 

# Configure plot appearance
ax.set_xlim(-1, 15)
ax.set_ylim(100, 800)
ax.set_xlabel('pH', fontsize=12)
ax.set_ylabel('Conductivity', fontsize=12)
ax.set_title('Water Quality: pH vs Conductivity', fontsize=14)
ax.grid(True, alpha=0.3)

# Add color legend for potability classification
legend_elements = [Patch(facecolor='red', label='Not Potable (0)'),
                   Patch(facecolor='blue', label='Potable (1)')]
ax.legend(handles=legend_elements, loc='best')

# Display the plot
plt.tight_layout()
plt.show()


### 3.6. **pH vs. TOC:**

#### **Total Organic Carbon (TOC):**
#### TOC in water sources results from the decay of natural organic matter as well as synthetic sources. It measures the total amount of carbon in organic compounds in pure water. According to US EPA standards, TOC should be less than 2 mg/L in treated drinking water and less than 4 mg/L in source water.

In [ ]:
# Visualize pH vs Organic Carbon with water potability indicators

# Extract feature columns for visualization
x = df_imputed['ph']
y = df_imputed['Organic_carbon']
potability = df_imputed['Potability']

# Color mapping: red = non-potable water (0), blue = potable water (1)
colors = []

for p in potability:
    if p == 0:
        colors.append('red')
    else:
        colors.append('blue')

# Initialize figure and axis
fig, (ax) = plt.subplots(figsize=(10, 6))

# Define acceptable water quality thresholds
ph_max_potability = 8.5
ph_min_potability = 6.5  

organic_carbon_max_potability = 2

# Highlight non-potable pH zones (outside 6.5-8.5 range)
ax.axvspan(ph_max_potability, 15, alpha=0.15, color='gray', zorder=0)  # High pH zone
ax.axvspan(-1, ph_min_potability, alpha=0.15, color='gray', zorder=0)  # Low pH zone

# Highlight non-potable organic carbon zone (above 2 mg/L)
ax.axhspan(organic_carbon_max_potability, 30, alpha=0.3, color='gray', zorder=0)  # High organic carbon zone

# Draw pH threshold boundary lines
ax.axvline(x=ph_max_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_max_potability}')
ax.axvline(x=ph_min_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_min_potability}')

# Draw organic carbon threshold boundary line
ax.axhline(y=organic_carbon_max_potability, color='green', linestyle='--', linewidth=2, label=f'Organic Carbon = {organic_carbon_max_potability}')

# Scatter plot of all water samples
ax.scatter(x, y, marker='o', s=20, alpha=0.6, c=colors, zorder=2) 

# Configure plot appearance
ax.set_xlim(-1, 15)
ax.set_ylim(0, 30)
ax.set_xlabel('pH', fontsize=12)
ax.set_ylabel('Organic Carbon', fontsize=12)
ax.set_title('Water Quality: pH vs Organic Carbon', fontsize=14)
ax.grid(True, alpha=0.3)

# Add color legend for potability classification
legend_elements = [Patch(facecolor='red', label='Not Potable (0)'),
                   Patch(facecolor='blue', label='Potable (1)')]
ax.legend(handles=legend_elements, loc='best')

# Display the plot
plt.tight_layout()
plt.show()


### 3.7. **pH vs. THM:**

#### **Trihalomethanes (THM):**
#### These are chemical compounds that may be found in water treated with chlorine. Their concentration depends on the amount of organic matter, the amount of chlorine used, and the water temperature during treatment. THM levels up to 80 ppm are considered safe in drinking water.

In [ ]:
# Visualize pH vs Trihalomethanes with water potability indicators

# Extract feature columns for visualization
x = df_imputed['ph']
y = df_imputed['Trihalomethanes']
potability = df_imputed['Potability']

# Color mapping: red = non-potable water (0), blue = potable water (1)
colors = []

for p in potability:
    if p == 0:
        colors.append('red')
    else:
        colors.append('blue')

# Initialize figure and axis
fig, (ax) = plt.subplots(figsize=(10, 6))

# Define acceptable water quality thresholds
ph_max_potability = 8.5
ph_min_potability = 6.5  

trihalomethanes_max_potability = 80

# Highlight non-potable pH zones (outside 6.5-8.5 range)
ax.axvspan(ph_max_potability, 15, alpha=0.15, color='gray', zorder=0)  # High pH zone
ax.axvspan(-1, ph_min_potability, alpha=0.15, color='gray', zorder=0)  # Low pH zone

# Highlight non-potable trihalomethanes zone (above 80 µg/L)
ax.axhspan(trihalomethanes_max_potability, 130, alpha=0.3, color='gray', zorder=0)  # High trihalomethanes zone

# Draw pH threshold boundary lines
ax.axvline(x=ph_max_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_max_potability}')
ax.axvline(x=ph_min_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_min_potability}')

# Draw trihalomethanes threshold boundary line
ax.axhline(y=trihalomethanes_max_potability, color='green', linestyle='--', linewidth=2, label=f'Trihalomethanes = {trihalomethanes_max_potability}')

# Scatter plot of all water samples
ax.scatter(x, y, marker='o', s=20, alpha=0.6, c=colors, zorder=2) 

# Configure plot appearance
ax.set_xlim(-1, 15)
ax.set_ylim(0, 130)
ax.set_xlabel('pH', fontsize=12)
ax.set_ylabel('Trihalomethanes', fontsize=12)
ax.set_title('Water Quality: pH vs Trihalomethanes', fontsize=14)
ax.grid(True, alpha=0.3)

# Add color legend for potability classification
legend_elements = [Patch(facecolor='red', label='Not Potable (0)'),
                   Patch(facecolor='blue', label='Potable (1)')]
ax.legend(handles=legend_elements, loc='best')

# Display the plot
plt.tight_layout()
plt.show()


### 3.8. **pH vs. Turbidity:**

#### **Turbidity:**
#### This depends on the amount of suspended solid particles in the water. It measures light passage through the water and is used to evaluate effluent quality regarding colloidal substances. The average turbidity obtained at Wondo Genet University was 0.0098 NTU, which is less than the WHO recommended value of 5.000 NTU.

In [ ]:
# Visualize pH vs Turbidity with water potability indicators

# Extract feature columns for visualization
x = df_imputed['ph']
y = df_imputed['Turbidity']
potability = df_imputed['Potability']

# Color mapping: red = non-potable water (0), blue = potable water (1)
colors = []

for p in potability:
    if p == 0:
        colors.append('red')
    else:
        colors.append('blue')

# Initialize figure and axis
fig, (ax) = plt.subplots(figsize=(10, 6))

# Define acceptable water quality thresholds
ph_max_potability = 8.5
ph_min_potability = 6.5  

turbidity_max_potability = 5

# Highlight non-potable pH zones (outside 6.5-8.5 range)
ax.axvspan(ph_max_potability, 15, alpha=0.15, color='gray', zorder=0)  # High pH zone
ax.axvspan(-1, ph_min_potability, alpha=0.15, color='gray', zorder=0)  # Low pH zone

# Highlight non-potable turbidity zone (above 5 NTU)
ax.axhspan(turbidity_max_potability, 7, alpha=0.3, color='gray', zorder=0)  # High turbidity zone

# Draw pH threshold boundary lines
ax.axvline(x=ph_max_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_max_potability}')
ax.axvline(x=ph_min_potability, color='orange', linestyle='--', linewidth=2, label=f'pH = {ph_min_potability}')

# Draw turbidity threshold boundary line
ax.axhline(y=turbidity_max_potability, color='green', linestyle='--', linewidth=2, label=f'Turbidity = {turbidity_max_potability}')

# Scatter plot of all water samples
ax.scatter(x, y, marker='o', s=20, alpha=0.6, c=colors, zorder=2) 

# Configure plot appearance
ax.set_xlim(-1, 15)
ax.set_ylim(0, 7)
ax.set_xlabel('pH', fontsize=12)
ax.set_ylabel('Turbidity', fontsize=12)
ax.set_title('Water Quality: pH vs Turbidity', fontsize=14)
ax.grid(True, alpha=0.3)

# Add color legend for potability classification
legend_elements = [Patch(facecolor='red', label='Not Potable (0)'),
                   Patch(facecolor='blue', label='Potable (1)')]
ax.legend(handles=legend_elements, loc='best')

# Display the plot
plt.tight_layout()
plt.show()


## 4. Conclusion
#### The evidence and distributions observed during this analysis suggest that this dataset was likely synthetically generated rather than collected from real-world, natural water sources.